In [1]:
import sys
sys.path.insert(0, "..")

from src.models        import DeepONet, build_deeponet
from src.data          import ODEIterableDataset, LatinHypercubeSampler
from src.physics       import harm_osc
from src.training      import Trainer, build_dataloaders

import numpy as np
import matplotlib.pyplot as plt 
import torch
from torch.utils.data import DataLoader

# Training 

In [2]:
# Initialize Harmonic Oscillator Object 
k = 2.0
c = 0.5
system = harm_osc([k, c])

# Initialize Sampler Object 
sampler = LatinHypercubeSampler(

    dimensions=2,
    lows      = [-1.0, -1.0],
    highs     = [1.0, 1.0]
    
)


In [3]:
# Initialize Training and 

train_dataset    = ODEIterableDataset(size = 1000,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 10),
                                      output_mask  = np.array([0]))

val_dataset      = ODEIterableDataset(size = 10,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 10),
                                      output_mask  = np.array([0]))

In [4]:
# Set Up DeepONet configuration 

DEEPONET_CONFIG = {
    
    "hidden_size" : 32,
    "depth"       : 4,
    "latent_size" : 50,
    "input_size_b": 2,
    "input_size_t": 1,
    "output_size" : 1,
    "activation"  : "tanh",

}

# Initialize DeepONet network 

deeponet = build_deeponet(DEEPONET_CONFIG)

In [ ]:
# Set Up training Configuration as well as optimizer and loss functions

TRAIN_CONFIG = {

    "num_epochs"     : 80,
    "learning_rate"  : 0.0068741,
    "batch_size"     : 32,
    "num_workers"    : 2,
    "Save_model"     : True,
    "Save_directory" : "../weights/best_1d_osc.pth"
}

optimizer = torch.optim.Adam
loss_fn   = torch.nn.MSELoss()


In [6]:
# Initialize Train Object and Train 

trainer   = Trainer(
    model         = deeponet,
    train_dataset = train_dataset, 
    val_dataset   = val_dataset, 
    optimizer     = optimizer,
    loss_fn       = loss_fn, 
    train_cfg     = TRAIN_CONFIG,
    model_cfg     = DEEPONET_CONFIG
)

trainer.run()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/enricp/.netrc.
wandb: Currently logged in as: nikpursals to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training: 100%|██████████| 1/1 [00:01<00:00,  1.91s/it]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
